# SONATE-2 frames from SatNOGS DB
Every frame ground stations received during our three passes (fetched via the SatNOGS DB API).

In [1]:
import pandas as pd
pd.set_option("display.max_colwidth", 60)

df = pd.read_csv("data/satnogs/sonate2_frames.csv", parse_dates=["timestamp"], dtype={"observation_id": "Int64"})
print(df.shape)
df.head(10)

(2543, 12)


,sat_id,norad_cat_id,transmitter,app_source,decoded,frame,observer,timestamp,version,observation_id,station_id,associated_satellites
0,PBTZ-3525-0189-2410-5216,59112,NaN,network,NaN,86A240404040E088A060A69CB06103F02700353618000864EC4B0073...,DK0SB-VHF/UHF-Hi-JO31ok,2026-09-21 16:18:48+00:00,NaN,15039241,1664.0,['YFDH-6340-5343-3152-5493']
1,PBTZ-3525-0189-2410-5216,59112,NaN,network,NaN,86A240404040E088A060A69CB06103F02700313318000864EC480073...,DK0SB-VHF/UHF-Hi-JO31ok,2026-09-21 16:18:42+00:00,NaN,15039241,1664.0,['YFDH-6340-5343-3152-5493']
2,PBTZ-3525-0189-2410-5216,59112,NaN,network,NaN,86A240404040E088A060A69CB06103F027002F3118000864EC460073...,DK0SB-VHF/UHF-Hi-JO31ok,2026-09-21 16:18:38+00:00,NaN,15039241,1664.0,['YFDH-6340-5343-3152-5493']
3,PBTZ-3525-0189-2410-5216,59112,NaN,network,NaN,86A240404040E088A060A69CB06103F027002D2F18000864EC440073...,DK0SB-VHF/UHF-Hi-JO31ok,2026-09-21 16:18:34+00:00,NaN,15039241,1664.0,['YFDH-6340-5343-3152-5493']
4,PBTZ-3525-0189-2410-5216,59112,NaN,network,NaN,86A240404040E088A060A69CB06103F027002C2E18000864EC430073...,DK0SB-VHF/UHF-Hi-JO31ok,2026-09-21 16:18:32+00:00,NaN,15039241,1664.0,['YFDH-6340-5343-3152-5493']
5,PBTZ-3525-0189-2410-5216,59112,NaN,network,NaN,86A240404040E088A060A69CB06103F027002B2D18000864EC420073...,DK0SB-VHF/UHF-Hi-JO31ok,2026-09-21 16:18:30+00:00,NaN,15039241,1664.0,['YFDH-6340-5343-3152-5493']
6,PBTZ-3525-0189-2410-5216,59112,NaN,network,NaN,86A240404040E088A060A69CB06103F027002A2C18000864EC410073...,DK0SB-VHF/UHF-Hi-JO31ok,2026-09-21 16:18:28+00:00,NaN,15039241,1664.0,['YFDH-6340-5343-3152-5493']
7,PBTZ-3525-0189-2410-5216,59112,NaN,network,NaN,86A240404040E088A060A69CB06103F0270229AC1FFF9E0000000000...,DK0SB-VHF/UHF-Hi-JO31ok,2026-09-21 16:18:27+00:00,NaN,15039241,1664.0,['YFDH-6340-5343-3152-5493']
8,PBTZ-3525-0189-2410-5216,59112,NaN,network,NaN,86A240404040E088A060A69CB06103F0270228AB1800087AC550006C...,DK0SB-VHF/UHF-Hi-JO31ok,2026-09-21 16:18:27+00:00,NaN,15039241,1664.0,['YFDH-6340-5343-3152-5493']
9,PBTZ-3525-0189-2410-5216,59112,NaN,network,NaN,86A240404040E088A060A69CB06103F02700272B18000864EC400073...,DK0SB-VHF/UHF-Hi-JO31ok,2026-09-21 16:18:26+00:00,NaN,15039241,1664.0,['YFDH-6340-5343-3152-5493']


## Frames per pass and per ground station

In [2]:
df.groupby(["observation_id", "observer"]).size().rename("frames").to_frame()

,,frames
observation_id,observer,
15036321,DEX-ONE-JO20ul,1
15039241,DK0SB-VHF/UHF-Hi-JO31ok,382
15039527,Astronomiemuseum Turnstile VHF (Test UHF)-JO50oj,3
15039839,PE0SAT-21-JO21mr,63
15039848,PE0SAT-31-JO21mr,3
15039885,ISAE Supaero V/UHF-JN03rn,180
15052725,DK0SB-UHF-Omni-JO31ok,241
15052729,DK0SB-VHF/UHF-Hi-JO31ok,602
15058812,PE0SAT-21-JO21mr,238


## Decode the AX.25 header
Each `frame` is hex. The first 14 bytes are the destination and source callsigns (each character shifted left by 1 bit).

In [3]:
def callsign(b):
    call = "".join(chr(c >> 1) for c in b[:6]).strip()
    ssid = (b[6] >> 1) & 0x0F
    return f"{call}-{ssid}" if ssid else call

raw = df["frame"].apply(bytes.fromhex)
# A few entries from other stations are junk (1-10 bytes), too short to hold a header
ok = raw.apply(len) >= 16
df["dst"] = raw[ok].apply(lambda b: callsign(b[0:7]))
df["src"] = raw[ok].apply(lambda b: callsign(b[7:14]))
df["length"] = raw.apply(len)
df["payload_hex"] = raw.apply(lambda b: b[16:].hex())   # after 14 address bytes + control + PID

df[["timestamp", "observer", "src", "dst", "length", "payload_hex"]].head(10)

,timestamp,observer,src,dst,length,payload_hex
0,2026-09-21 16:18:48+00:00,DK0SB-VHF/UHF-Hi-JO31ok,DP0SNX,CQ,146,2700353618000864ec4b00736ab158e98218540e303f428000000000...
1,2026-09-21 16:18:42+00:00,DK0SB-VHF/UHF-Hi-JO31ok,DP0SNX,CQ,146,2700313318000864ec4800736ab158e382185408303f428000000000...
2,2026-09-21 16:18:38+00:00,DK0SB-VHF/UHF-Hi-JO31ok,DP0SNX,CQ,146,27002f3118000864ec4600736ab158df82185404303f428000000000...
3,2026-09-21 16:18:34+00:00,DK0SB-VHF/UHF-Hi-JO31ok,DP0SNX,CQ,146,27002d2f18000864ec4400736ab158db82185400303f428000000000...
4,2026-09-21 16:18:32+00:00,DK0SB-VHF/UHF-Hi-JO31ok,DP0SNX,CQ,146,27002c2e18000864ec4300736ab158d9821853fe303f428000000000...
5,2026-09-21 16:18:30+00:00,DK0SB-VHF/UHF-Hi-JO31ok,DP0SNX,CQ,146,27002b2d18000864ec4200736ab158d7821853fc303f428000000000...
6,2026-09-21 16:18:28+00:00,DK0SB-VHF/UHF-Hi-JO31ok,DP0SNX,CQ,146,27002a2c18000864ec4100736ab158d5821853fa303f428000000000...
7,2026-09-21 16:18:27+00:00,DK0SB-VHF/UHF-Hi-JO31ok,DP0SNX,CQ,97,270229ac1fff9e000000000000008000000000000000000000000000...
8,2026-09-21 16:18:27+00:00,DK0SB-VHF/UHF-Hi-JO31ok,DP0SNX,CQ,271,270228ab1800087ac550006c6ab158d39efffc00000001fdfc000000...
9,2026-09-21 16:18:26+00:00,DK0SB-VHF/UHF-Hi-JO31ok,DP0SNX,CQ,146,2700272b18000864ec4000736ab158d3821853f8303f428000000000...


In [4]:
print("Unique frames:", df["frame"].nunique(), "of", len(df), "rows")
print("Too short to decode:", (~ok).sum())
df["src"].value_counts(dropna=False)

Unique frames: 1127 of 2543 rows
Too short to decode: 6


src
DP0SNX        2525
DL7NDR           7
NaN              6
pBH\t%-11       1
QV
4O-12        1
5%7-1         1
\*RH
]-12        1
c)z1-12        1
Name: count, dtype: int64